# Kendall correlation benchmark

For every method and synthetic dataset, this notebook calculates the mean
gene-wise Kendall correlation between the method score and the simulated
spatial-signal strength. Dataset names are discovered from the method folders.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

input_dir = Path(".")
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)

method_scores = {
    "morans": ("I", 1),
    "spatialde": ("qval", -1),
    "smash": ("Adjusted p-val", -1),
    "hotspot": ("Pval", -1),
    "scgco": ("fdr", -1),
    "somde": ("Pval", -1),
    "spagft": ("gft_score", 1),
    "svgbit": ("AI", 1),
}

method_order = [
    "hotspot",
    "morans",
    "scgco",
    "smash",
    "somde",
    "spagft",
    "spatialde",
    "svgbit",
]

method_labels = {
    "hotspot": "Hotspot",
    "morans": "Moran's I",
    "scgco": "scGCO",
    "smash": "SMASH",
    "somde": "SOMDE",
    "spagft": "SpaGFT",
    "spatialde": "SpatialDE",
    "svgbit": "SVGBit",
}

In [ ]:
def discover_datasets():
    """Return all CSV basenames found in the configured method folders."""
    datasets = set()

    for method in method_scores:
        method_dir = input_dir / method
        if method_dir.is_dir():
            datasets.update(path.stem for path in method_dir.glob("*.csv"))

    if not datasets:
        raise FileNotFoundError(
            f"No method result CSV files were found below {input_dir.resolve()}."
        )

    return sorted(datasets)


def compute_correlation(data, method):
    """Calculate mean gene-wise Kendall correlation for one method."""
    score_column, direction = method_scores[method]
    required = {"gene", "spatial_var", score_column}
    missing = required.difference(data.columns)

    if missing:
        raise ValueError(f"{method} result is missing columns: {sorted(missing)}")

    values = data[["gene", "spatial_var", score_column]].copy()
    values["spatial_var"] = pd.to_numeric(
        values["spatial_var"], errors="coerce"
    )
    values["score"] = direction * pd.to_numeric(
        values[score_column], errors="coerce"
    )
    values = values.dropna(subset=["gene", "spatial_var", "score"])

    correlations = [
        group["score"].corr(group["spatial_var"], method="kendall")
        for _, group in values.groupby("gene", observed=True)
    ]

    return pd.Series(correlations, dtype=float).mean()


def collect_correlations(datasets):
    records = []

    for dataset in datasets:
        for method in method_scores:
            path = input_dir / method / f"{dataset}.csv"

            if not path.exists():
                continue

            records.append({
                "dataset": dataset,
                "method": method,
                "correlation": compute_correlation(
                    pd.read_csv(path), method
                ),
            })

    results = pd.DataFrame(records)

    if results.empty:
        raise ValueError("No valid method results were processed.")

    results["correlation"] = results["correlation"].fillna(0)
    return results

In [ ]:
datasets = discover_datasets()
results = collect_correlations(datasets)

results.to_csv(
    output_dir / "kendall_results.csv",
    index=False,
)

available_methods = [
    method for method in method_order
    if method in results["method"].unique()
]

plt.figure(figsize=(9, 5))
sns.boxplot(
    data=results,
    x="method",
    y="correlation",
    order=available_methods,
    showfliers=False,
    width=0.6,
    boxprops={"linewidth": 1.5},
    whiskerprops={"linewidth": 1.5},
    capprops={"linewidth": 1.5},
    medianprops={"color": "black", "linewidth": 2},
)
plt.xlabel("Method")
plt.ylabel("Kendall correlation")
plt.xticks(
    range(len(available_methods)),
    [method_labels[method] for method in available_methods],
    rotation=45,
    ha="right",
)
plt.title("Correlation with simulated spatial signal")
plt.tight_layout()
plt.savefig(
    output_dir / "kendall_correlation_boxplot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

results